In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

# 1. Model klasörünü kontrol et
model_klasoru = "/content/drive/MyDrive/data/1d-cnn_best_2_shap/models/"
print("--- MODEL KLASÖRÜ KONTROLÜ ---")
if os.path.exists(model_klasoru):
    print("Klasör bulundu! İçindeki dosyalar:")
    print(os.listdir(model_klasoru))
else:
    print(f"HATA: '{model_klasoru}' yolu bulunamadı. Klasör isimlerini kontrol et.")

print("\n" + "="*50 + "\n")

# 2. CSV klasörünü kontrol et
csv_klasoru = "/content/drive/MyDrive/data/ai-ready-data/"
print("--- CSV KLASÖRÜ KONTROLÜ ---")
if os.path.exists(csv_klasoru):
    print("Klasör bulundu! İçindeki dosyalar:")
    print(os.listdir(csv_klasoru))
else:
    print(f"HATA: '{csv_klasoru}' yolu bulunamadı. Klasör isimlerini kontrol et.")

--- MODEL KLASÖRÜ KONTROLÜ ---
Klasör bulundu! İçindeki dosyalar:
['model_architecture.txt', 'fold_1_best_model.keras', 'fold_2_best_model.keras', 'fold_3_best_model.keras', 'fold_4_best_model.keras', 'fold_5_best_model.keras', 'fold_6_best_model.keras', 'fold_7_best_model.keras', 'fold_8_best_model.keras', 'fold_9_best_model.keras', 'fold_10_best_model.keras', 'best_overall_model.keras', 'best_weights.weights.h5']


--- CSV KLASÖRÜ KONTROLÜ ---
Klasör bulundu! İçindeki dosyalar:
['train_scaled.csv', 'test_scaled.csv', '.ipynb_checkpoints', 'shap_outputs', 'shap_outputs_2', 'test_selected_shap.csv', 'train_selected_shap.csv', 'shap_lime_final_outputs', 'ig_final_outputs', 'shap_lime_yeni_analiz']


In [3]:
import tensorflow as tf
import pandas as pd
import numpy as np
import time
import os
import psutil

# ---------------------------------------------------------
# 0. TENSORFLOW VRAM YÖNETİMİ (MEMORY GROWTH)
# ---------------------------------------------------------
# Bu ayar, TensorFlow'un tüm GPU belleğini rezerve etmesini engeller.
# Yalnızca o an ihtiyaç duyulan kadar VRAM tahsis edilir.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU Memory Growth AKTİF: TensorFlow sadece gerektiği kadar VRAM kullanacak.\n")
    except RuntimeError as e:
        print(f"Memory Growth ayarlanırken hata: {e}")
else:
    print("⚠️ GPU bulunamadı, CPU üzerinden devam ediliyor.\n")


def evaluate_keras_model_streaming(model_path, csv_path, target_column=None, num_packets=1000):
    """
    Keras modelinin GERÇEK ZAMANLI (Streaming) ağ trafiği altındaki performansını ölçer.

    Parametreler:
    - num_packets: Arka arkaya simüle edilecek/işlenecek tekil ağ paketi (satır) sayısı.
    """
    print("="*50)
    print("🧠 ÜRETİM ORTAMI (PRODUCTION) CANLI AKIŞ RAPORU")
    print("="*50)

    # ---------------------------------------------------------
    # 1. VERİ YÜKLEME VE HAZIRLAMA
    # ---------------------------------------------------------
    print(f"Veri yükleniyor: {csv_path}...")
    df = pd.read_csv(csv_path)

    if target_column and target_column in df.columns:
        df = df.drop(columns=[target_column])

    # İstenen paket sayısı CSV'deki toplam satırdan büyükse sınırla
    num_packets = min(num_packets, len(df))

    # Keras için veriyi tf.Tensor'a çeviriyoruz
    input_data = tf.convert_to_tensor(df.values, dtype=tf.float32)

    # ---------------------------------------------------------
    # 2. MODEL YÜKLEME VE BOYUT BİLGİLERİ
    # ---------------------------------------------------------
    print(f"Model yükleniyor: {model_path}...")
    model = tf.keras.models.load_model(model_path)

    # Disk Boyutu ve Parametreler
    total_params = model.count_params()
    file_size_mb = os.path.getsize(model_path) / (1024 * 1024)

    print(f"Model Disk Boyutu        : {file_size_mb:.4f} MB")
    print(f"Toplam Parametre Sayısı  : {total_params:,}")

    # ---------------------------------------------------------
    # 3. GERÇEK ZAMANLI AĞ TRAFİĞİ SİMÜLASYONU (STREAMING)
    # ---------------------------------------------------------
    print(f"\n⏱️ Canlı Trafik Simülasyonu Başlıyor ({num_packets} paket)...")

    # tf.function, Python ek yükünü kaldırarak matris çarpımını maksimum hıza ulaştırır
    @tf.function
    def fast_predict(x):
        return model(x, training=False)

    # Donanımsal Isınma (Warm-up) - Keras grafiğini derlemek için
    for _ in range(10):
        _ = fast_predict(input_data[0:1])

    # --- SİMÜLASYON BAŞLANGICI ---
    start_time = time.perf_counter()

    # Paketleri (satırları) tıpkı bir ağ kartından geliyormuş gibi TEK TEK veriyoruz
    for i in range(num_packets):
        _ = fast_predict(input_data[i:i+1])

    end_time = time.perf_counter()
    # --- SİMÜLASYON BİTİŞİ ---

    total_time = end_time - start_time
    avg_latency_ms = (total_time / num_packets) * 1000
    throughput = num_packets / total_time

    print(f"Simüle Edilen Paket Sayısı : {num_packets}")
    print(f"Ortalama Paket Gecikmesi   : {avg_latency_ms:.4f} ms (Milisaniye)")
    print(f"Sistem Kapasitesi          : Saniyede {throughput:.2f} paket işlenebilir")

    # ---------------------------------------------------------
    # 4. RAM VE VRAM (GPU) TÜKETİMİ
    # ---------------------------------------------------------
    print("\n📊 Kaynak Tüketimi (Resource Usage)...")
    process = psutil.Process(os.getpid())
    memory_info = process.memory_info()
    ram_usage_mb = memory_info.rss / (1024 * 1024)

    print(f"Mevcut Süreç RAM Kullanımı : {ram_usage_mb:.2f} MB")

    if gpus:
        try:
            memory_info = tf.config.experimental.get_memory_info('GPU:0')
            vram_current = memory_info['current'] / (1024 * 1024)
            vram_peak = memory_info['peak'] / (1024 * 1024)
            print(f"GPU VRAM Kullanımı (Mevcut): {vram_current:.2f} MB")
            print(f"GPU VRAM Kullanımı (Zirve) : {vram_peak:.2f} MB")
        except Exception as e:
            print("VRAM bilgisi alınamadı (TensorFlow sürümü desteklemiyor olabilir).")

    print("="*50)


# ---------------------------------------------------------
# KULLANIM
# ---------------------------------------------------------
if __name__ == "__main__":
    MODEL_DOSYASI = "/content/drive/MyDrive/data/1d-cnn_best_2_shap/models/best_overall_model.keras"
    CSV_DOSYASI = "/content/drive/MyDrive/data/ai-ready-data/test_selected_shap.csv"

    HEDEF_SUTUN = "Label"

    if os.path.exists(MODEL_DOSYASI) and os.path.exists(CSV_DOSYASI):
        # num_packets değerini buradan istediğin gibi değiştirebilirsin.
        # Örneğin: 100, 5000, 10000 gibi değerler vererek sistemin uzun süreli akışa tepkisini ölçebilirsin.
        evaluate_keras_model_streaming(MODEL_DOSYASI, CSV_DOSYASI, target_column=HEDEF_SUTUN, num_packets=10000)
    else:
        print(f"HATA: Dosyalar bulunamadı.\n1. Model Yolu: {MODEL_DOSYASI}\n2. CSV Yolu: {CSV_DOSYASI}")

✅ GPU Memory Growth AKTİF: TensorFlow sadece gerektiği kadar VRAM kullanacak.

🧠 ÜRETİM ORTAMI (PRODUCTION) CANLI AKIŞ RAPORU
Veri yükleniyor: /content/drive/MyDrive/data/ai-ready-data/test_selected_shap.csv...
Model yükleniyor: /content/drive/MyDrive/data/1d-cnn_best_2_shap/models/best_overall_model.keras...
Model Disk Boyutu        : 0.5380 MB
Toplam Parametre Sayısı  : 41,278

⏱️ Canlı Trafik Simülasyonu Başlıyor (10000 paket)...
Simüle Edilen Paket Sayısı : 10000
Ortalama Paket Gecikmesi   : 1.4579 ms (Milisaniye)
Sistem Kapasitesi          : Saniyede 685.93 paket işlenebilir

📊 Kaynak Tüketimi (Resource Usage)...
Mevcut Süreç RAM Kullanımı : 1878.95 MB
GPU VRAM Kullanımı (Mevcut): 128.52 MB
GPU VRAM Kullanımı (Zirve) : 178.08 MB
